# Explainability Evaluation - Faithfulness

## Testing the Attention based Explainability and SHAP on Faithfulness

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Connect and Access Git Repository

import os
import sys
import json
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"

token = getpass("GitHub token: ")

REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
Cloning into '/content/bachelor-thesis'...
remote: Enumerating objects: 743, done.
remote: Counting objects: 100% (258/258), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 743 (delta 167), reused 127 (delta 70), pack-reused 485 (from 2)
Receiving objects: 100% (743/743), 37.73 MiB | 17.62 MiB/s, done.
Resolving deltas: 100% (407/407), done.
/content/bachelor-thesis
Repo exists: True


In [21]:
# Imports
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
import shap

# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))


from utils.dataset import PTBXLDataset
from utils.seed import set_seed, RANDOM_SEED
from explainability.anoformer_attention import AnoFormerExplainability
from explainability.tranad_attention import TranADExplainability
from explainability.shap_wrappers import (
    AnoFormerAnomalyScoreWrapper,
    TranADAnomalyScoreWrapper,
)
from explainability.evaluation import FaithfulnessEvaluation
from explainability.sample_selection import load_samples_from_dataset

from models.anoformer import AnoFormerMainModel
from models.tranad import TranADMainModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [4]:
# Paths to Directories
# Load processed data and pretrained models

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"
SHAP_RESULTS_DIR = RESULTS_DIR / "shap"

MODEL_DIR = RESULTS_DIR / "models"
ANOFORMER_MODEL_PATH = MODEL_DIR / "anoformer" / "anoformer_40epochs_larger_capacity_2.pt"
TRANAD_MODEL_PATH = MODEL_DIR / "tranad" / "tranad_20epochs_larger_capacity.pt"

FAITHFULNESS_DIR = RESULTS_DIR / "evaluation" / "faithfulness"
FAITHFULNESS_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Load my preprocessed training and test datasets

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2198, 1000, 12) (2198,)


In [6]:
# select only normal ecgs from Training set
normal_train_mask = y_train == 0
X_train_normal = X_train[normal_train_mask]
y_train_normal = y_train[normal_train_mask]

# Datasets
train_dataset = PTBXLDataset(
    X_train_normal,
    y_train_normal,
    normalize=True
)
test_dataset = PTBXLDataset(
    X_test,
    y_test,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

In [7]:
def make_single_ecg_loader(ecg):
    """
    Create a DataLoader containing exactly one ECG
    Expected ECG shape: (1, 1000, 12)
    """

    dummy_label = torch.zeros(ecg.shape[0], dtype=torch.long)
    dataset = TensorDataset(ecg, dummy_label)

    return DataLoader(dataset, batch_size=1, shuffle=False)

In [8]:
# Load fixed evaluation samples

EvaluationSamplePath = (
    RESULTS_DIR
    / "evaluation"
    / "evaluation_samples.npz"
)

EvaluationSamples = load_samples_from_dataset(
    sample_path=EvaluationSamplePath,
    test_dataset=test_dataset,
    device=None,
)

print("Number of evaluation samples:", len(EvaluationSamples))

for sample in EvaluationSamples[:5]:
    print(
        "Index:",
        sample["index"],
        "Case:",
        sample["case_type"],
        "Label:",
        sample["label"],
        "Shape:",
        sample["ecg"].shape,
    )

Number of evaluation samples: 20
Index: 2154 Case: TN Label: 0 Shape: torch.Size([1, 1000, 12])
Index: 1647 Case: TN Label: 0 Shape: torch.Size([1, 1000, 12])
Index: 161 Case: TN Label: 0 Shape: torch.Size([1, 1000, 12])
Index: 1394 Case: TN Label: 0 Shape: torch.Size([1, 1000, 12])
Index: 929 Case: TN Label: 0 Shape: torch.Size([1, 1000, 12])


In [9]:
# Load previously calculated Anomaly Thresholds
# these were calculated based on the evaluation set

filepath = MODEL_DIR / "chosen_thresholds.json"

with open(filepath, "r") as f:
    thresholds = json.load(f)

AnoFormerChosenThreshold = thresholds["anoformer"]
TranADChosenThreshold = thresholds["tranad"]

In [10]:
# Create the Evaluator

FaithfulnessEvaluator = FaithfulnessEvaluation(
    top_fraction=0.10,
    epsilon=1e-8,
)

# AnoFormer Evaluation

In [11]:
# Create AnoFormer Model

AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

# Load the pretrained model
AnoFormer.Load(str(ANOFORMER_MODEL_PATH))

# Create Explainer to extract Attention Values
AnoFormerExplainer = AnoFormerExplainability(AnoFormer)

Model loaded from: /content/drive/MyDrive/ECG data/results/models/anoformer/anoformer_40epochs_larger_capacity_2.pt


In [12]:
def anoformer_score_function(ecg):
    """
    Return the AnoFormer anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = AnoFormer.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=AnoFormerChosenThreshold,
        NumMaskRuns=5,
        Seed=RANDOM_SEED,
    )

    return scores[0]

### Attention Evaluation

In [13]:
# First we collect the Attention values for the samples

AnoFormerAttentionImportanceValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(AnoFormer.Device)

    attention_importance = AnoFormerExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        BlockIndex=0,
        HeadIndex=None,
    )

    AnoFormerAttentionImportanceValues.append(
        attention_importance
    )

In [15]:
# Evaluate the Attention Faithfulness of all sample ECGs

AnoFormerAttentionFaithfulnessList = (
    FaithfulnessEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=AnoFormerAttentionImportanceValues,
        score_function=anoformer_score_function,
        model_name="AnoFormer",
        explanation_name="Attention"
    )
)

AnoFormerAttentionFaithfulnessResults = pd.DataFrame(
    AnoFormerAttentionFaithfulnessList
)

AnoFormerAttentionFaithfulnessResults

,Model,Explanation,Index,Case Type,Label,Original Score,Perturbed Score,Score Difference,Faithfulness Score
0,AnoFormer,Attention,2154,TN,0,0.113553,0.138499,0.024946,0.219688
1,AnoFormer,Attention,1647,TN,0,0.135731,0.099133,-0.036598,0.269636
2,AnoFormer,Attention,161,TN,0,0.141760,0.146045,0.004285,0.030227
3,AnoFormer,Attention,1394,TN,0,0.133735,0.094869,-0.038866,0.290618
4,AnoFormer,Attention,929,TN,0,0.121561,0.039250,-0.082311,0.677115
5,AnoFormer,Attention,911,TN,0,0.169591,0.061928,-0.107662,0.634837
6,AnoFormer,Attention,1528,TN,0,0.174179,0.172596,-0.001583,0.009088
7,AnoFormer,Attention,193,TN,0,0.190106,0.121218,-0.068888,0.362366
8,AnoFormer,Attention,383,TN,0,0.091696,0.089365,-0.002331,0.025418
9,AnoFormer,Attention,1878,TN,0,0.111542,0.063639,-0.047904,0.429466


In [16]:
# All results averaged into one faithfullness score per Case Type

AnoFormerAttentionFaithfulnessSummary = (
    AnoFormerAttentionFaithfulnessResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Faithfulness Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerAttentionFaithfulnessSummary

,Model,Explanation,Case Type,mean,std,count
0,AnoFormer,Attention,TN,0.294846,0.239207,10
1,AnoFormer,Attention,TP,0.456079,0.125415,10


In [17]:
# Save the Results

AnoFormerAttentionFaithfulnessResults.to_csv(
    FAITHFULNESS_DIR / "anoformer_attention_faithfulness_results.csv",
    index=False,
)

AnoFormerAttentionFaithfulnessSummary.to_csv(
    FAITHFULNESS_DIR / "anoformer_attention_faithfulness_summary.csv",
    index=False,
)

## SHAP Evaluation

In [18]:
# Load the background indices used for SHAP explanations

background_indices = np.load(
    SHAP_RESULTS_DIR / "medium_background_indices.npy"
)

background_ECGs = torch.as_tensor(
    X_train[background_indices],
    dtype=torch.float32,
    device=DEVICE,
)

In [19]:
# Save the results because I can use them later

ANOFORMER_SHAP_DIR = (
    SHAP_RESULTS_DIR
    / "anoformer"
)

ANOFORMER_SHAP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

AnoFormerEvaluationSHAPPath = (
    ANOFORMER_SHAP_DIR
    / "anoformer_evaluation_shap_values.npy"
)

In [22]:
# Create the same Anomaly Scpre Wrapper for AnoFormer that was used during SHAP explanations

AnoFormerScoreWrapper = AnoFormerAnomalyScoreWrapper(
    anoformer_model=AnoFormer,
    window_size=1000,
    masking_rate=None,
    num_masks=1,
    seed=RANDOM_SEED,
).to(DEVICE)

AnoFormerScoreWrapper.eval()

# create Explainer
AnoFormerSHAPExplainer = shap.GradientExplainer(
    AnoFormerScoreWrapper,
    background_ECGs,
)

In [23]:
# Compute SHAP values for sample ECGs
AnoFormerSHAPValueList = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(
        AnoFormer.Device
    )

    shap_values = AnoFormerSHAPExplainer.shap_values(ecg)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(shap_values)

    if shap_values.ndim == 4:
        shap_values = shap_values[..., 0]

    shap_values = shap_values[0]

    AnoFormerSHAPValueList.append(shap_values.astype(np.float32))

    print(
        "Computed AnoFormer SHAP for index:",
        sample["index"],
        "case:",
        sample["case_type"],
        "shape:",
        shap_values.shape,
    )

AnoFormerSHAPValues = np.stack(AnoFormerSHAPValueList, axis=0)

# For later use
np.save(AnoFormerEvaluationSHAPPath, AnoFormerSHAPValues)

print(
    "Saved AnoFormer SHAP values:",
    AnoFormerEvaluationSHAPPath,
    AnoFormerSHAPValues.shape,
)

Computed AnoFormer SHAP for index: 2154 case: TN shape: (1000, 12)
Computed AnoFormer SHAP for index: 1647 case: TN shape: (1000, 12)
Computed AnoFormer SHAP for index: 161 case: TN shape: (1000, 12)
Computed AnoFormer SHAP for index: 1394 case: TN shape: (1000, 12)
Computed AnoFormer SHAP for index: 929 case: TN shape: (1000, 12)
Computed AnoFormer SHAP for index: 911 case: TN shape: (1000, 12)
Computed AnoFormer SHAP for index: 1528 case: TN shape: (1000, 12)
Computed AnoFormer SHAP for index: 193 case: TN shape: (1000, 12)
Computed AnoFormer SHAP for index: 383 case: TN shape: (1000, 12)
Computed AnoFormer SHAP for index: 1878 case: TN shape: (1000, 12)
Computed AnoFormer SHAP for index: 1134 case: TP shape: (1000, 12)
Computed AnoFormer SHAP for index: 1854 case: TP shape: (1000, 12)
Computed AnoFormer SHAP for index: 102 case: TP shape: (1000, 12)
Computed AnoFormer SHAP for index: 1310 case: TP shape: (1000, 12)
Computed AnoFormer SHAP for index: 1462 case: TP shape: (1000, 12)
C

In [24]:
# Convert AnoFormer SHAP values to temporal importance

AnoFormerSHAPImportanceValues = []

for shap_values in AnoFormerSHAPValues:
    temporal_importance = np.abs(
        shap_values
    ).sum(axis=1)

    AnoFormerSHAPImportanceValues.append(
        temporal_importance
    )

print(
    "Number of AnoFormer SHAP importance arrays:",
    len(AnoFormerSHAPImportanceValues),
)

print(
    "Shape of first AnoFormer SHAP importance:",
    AnoFormerSHAPImportanceValues[0].shape,
)

Number of AnoFormer SHAP importance arrays: 20
Shape of first AnoFormer SHAP importance: (1000,)


In [26]:
AnoFormerSHAPFaithfulnessList = (
    FaithfulnessEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=AnoFormerSHAPImportanceValues,
        score_function=anoformer_score_function,
        model_name="AnoFormer",
        explanation_name="SHAP"
    )
)

AnoFormerSHAPFaithfulnessResults = pd.DataFrame(
    AnoFormerSHAPFaithfulnessList
)

AnoFormerSHAPFaithfulnessResults

,Model,Explanation,Index,Case Type,Label,Original Score,Perturbed Score,Score Difference,Faithfulness Score
0,AnoFormer,SHAP,2154,TN,0,0.113553,0.162001,0.048448,0.426659
1,AnoFormer,SHAP,1647,TN,0,0.135731,0.086007,-0.049724,0.366342
2,AnoFormer,SHAP,161,TN,0,0.141760,0.158067,0.016307,0.115034
3,AnoFormer,SHAP,1394,TN,0,0.133735,0.080484,-0.053251,0.398182
4,AnoFormer,SHAP,929,TN,0,0.121561,0.038479,-0.083082,0.683457
5,AnoFormer,SHAP,911,TN,0,0.169591,0.047941,-0.121649,0.717311
6,AnoFormer,SHAP,1528,TN,0,0.174179,0.143218,-0.030961,0.177752
7,AnoFormer,SHAP,193,TN,0,0.190106,0.174611,-0.015495,0.081507
8,AnoFormer,SHAP,383,TN,0,0.091696,0.128132,0.036436,0.397358
9,AnoFormer,SHAP,1878,TN,0,0.111542,0.048228,-0.063314,0.567623


In [27]:
AnoFormerSHAPFaithfulnessSummary = (
    AnoFormerSHAPFaithfulnessResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Faithfulness Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerSHAPFaithfulnessSummary

,Model,Explanation,Case Type,mean,std,count
0,AnoFormer,SHAP,TN,0.393123,0.221748,10
1,AnoFormer,SHAP,TP,0.376761,0.187494,10


In [28]:
# Save the Results

AnoFormerSHAPFaithfulnessResults.to_csv(
    FAITHFULNESS_DIR / "anoformer_shap_faithfulness_results.csv",
    index=False,
)

AnoFormerSHAPFaithfulnessSummary.to_csv(
    FAITHFULNESS_DIR / "anoformer_shap_faithfulness_summary.csv",
    index=False,
)

# TranAD Evaluation

In [29]:
# Create TranAD Model

TranAD = TranADMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    ForwardDimension=128,
    DropoutRate=0.1,
    LearningRate=1e-4,
    FocusGrowthRate=1.05,
    InitialFocusScore=1.0,
    MaxFocusScore=5.0,
    AnomalyPercentile=95.0,
)

# Load the pretrained model
TranAD.Load(str(TRANAD_MODEL_PATH))

# Create Explainer to extract Attention Values
TranADExplainer = TranADExplainability(TranAD)


TranADModel: 268,376 Parameter | d_model=64, H=4, Enc=2, Dec×2=2
Model loaded  /content/drive/MyDrive/ECG data/results/models/tranad/tranad_20epochs_larger_capacity.pt


In [30]:
def tranad_score_function(ecg):
    """
    Return the TranAD anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = TranAD.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=TranADChosenThreshold,
    )

    return scores[0]

## Attention Evaluation

In [31]:
# Extract Attention Values of Original ECGs

TranADAttentionImportanceValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(TranAD.Device)

    attention_importance = TranADExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        LayerIndex=0,
        HeadIndex=None,
    )

    TranADAttentionImportanceValues.append(
        attention_importance
    )

In [33]:
# Evaluate

TranADAttentionFaithfulnessList = (
    FaithfulnessEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=TranADAttentionImportanceValues,
        score_function=tranad_score_function,
        model_name="TranAD",
        explanation_name="Attention",
    )
)

TranADAttentionFaithfulnessResults = pd.DataFrame(
    TranADAttentionFaithfulnessList
)

TranADAttentionFaithfulnessResults

,Model,Explanation,Index,Case Type,Label,Original Score,Perturbed Score,Score Difference,Faithfulness Score
0,TranAD,Attention,2154,TN,0,0.005808,0.004232,-0.001576,0.271324
1,TranAD,Attention,1647,TN,0,0.009400,0.009270,-0.000130,0.013882
2,TranAD,Attention,161,TN,0,0.006775,0.006421,-0.000353,0.052121
3,TranAD,Attention,1394,TN,0,0.007250,0.007102,-0.000148,0.020359
4,TranAD,Attention,929,TN,0,0.004773,0.002959,-0.001814,0.380056
5,TranAD,Attention,911,TN,0,0.006403,0.003183,-0.003220,0.502868
6,TranAD,Attention,1528,TN,0,0.006568,0.006395,-0.000174,0.026433
7,TranAD,Attention,193,TN,0,0.009590,0.009093,-0.000497,0.051792
8,TranAD,Attention,383,TN,0,0.005922,0.006013,0.000091,0.015449
9,TranAD,Attention,1878,TN,0,0.006818,0.004114,-0.002704,0.396555


In [34]:
TranADAttentionFaithfulnessSummary = (
    TranADAttentionFaithfulnessResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Faithfulness Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADAttentionFaithfulnessSummary

,Model,Explanation,Case Type,mean,std,count
0,TranAD,Attention,TN,0.173084,0.193090,10
1,TranAD,Attention,TP,0.509286,0.246041,10


In [35]:
# Save the Results

TranADAttentionFaithfulnessResults.to_csv(
    FAITHFULNESS_DIR / "tranad_attention_faithfulness_results.csv",
    index=False,
)

TranADAttentionFaithfulnessSummary.to_csv(
    FAITHFULNESS_DIR / "tranad_attention_faithfulness_summary.csv",
    index=False,
)

## SHAP Evaluation

In [36]:
# Save the results because I can use them later

TRANAD_SHAP_DIR = (
    SHAP_RESULTS_DIR
    / "tranad"
)

TRANAD_SHAP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TranADEvaluationSHAPPath = (
    TRANAD_SHAP_DIR
    / "tranad_evaluation_shap_values.npy"
)

In [37]:
TranADScoreWrapper = TranADAnomalyScoreWrapper(TranAD)

# create Explainer
TranADSHAPExplainer = shap.GradientExplainer(
    TranADScoreWrapper,
    background_ECGs,
)

In [38]:
# Compute SHAP values for sample ECGs
TranADSHAPValueList = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(
        TranAD.Device
    )

    shap_values = TranADSHAPExplainer.shap_values(ecg)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(shap_values)

    if shap_values.ndim == 4:
        shap_values = shap_values[..., 0]

    shap_values = shap_values[0]

    TranADSHAPValueList.append(shap_values.astype(np.float32))

    print(
        "Computed TranAD SHAP for index:",
        sample["index"],
        "case:",
        sample["case_type"],
        "shape:",
        shap_values.shape,
    )

TranADSHAPValues = np.stack(TranADSHAPValueList, axis=0)

# For later use
np.save(TranADEvaluationSHAPPath, TranADSHAPValues)

print(
    "Saved TranAD SHAP values:",
    TranADEvaluationSHAPPath,
    TranADSHAPValues.shape,
)

Computed TranAD SHAP for index: 2154 case: TN shape: (1000, 12)
Computed TranAD SHAP for index: 1647 case: TN shape: (1000, 12)
Computed TranAD SHAP for index: 161 case: TN shape: (1000, 12)
Computed TranAD SHAP for index: 1394 case: TN shape: (1000, 12)
Computed TranAD SHAP for index: 929 case: TN shape: (1000, 12)
Computed TranAD SHAP for index: 911 case: TN shape: (1000, 12)
Computed TranAD SHAP for index: 1528 case: TN shape: (1000, 12)
Computed TranAD SHAP for index: 193 case: TN shape: (1000, 12)
Computed TranAD SHAP for index: 383 case: TN shape: (1000, 12)
Computed TranAD SHAP for index: 1878 case: TN shape: (1000, 12)
Computed TranAD SHAP for index: 1134 case: TP shape: (1000, 12)
Computed TranAD SHAP for index: 1854 case: TP shape: (1000, 12)
Computed TranAD SHAP for index: 102 case: TP shape: (1000, 12)
Computed TranAD SHAP for index: 1310 case: TP shape: (1000, 12)
Computed TranAD SHAP for index: 1462 case: TP shape: (1000, 12)
Computed TranAD SHAP for index: 1041 case: TP 

In [39]:
TranADSHAPImportanceValues = []

for shap_values in TranADSHAPValues:
    temporal_importance = np.abs(
        shap_values
    ).sum(axis=1)

    TranADSHAPImportanceValues.append(
        temporal_importance
    )
print(
    "Number of TranAD SHAP importance arrays:",
    len(TranADSHAPImportanceValues),
)

print(
    "Shape of first TranAD SHAP importance:",
    TranADSHAPImportanceValues[0].shape,
)

Number of TranAD SHAP importance arrays: 20
Shape of first TranAD SHAP importance: (1000,)


In [40]:
TranADSHAPFaithfulnessList = (
    FaithfulnessEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=TranADSHAPImportanceValues,
        score_function=tranad_score_function,
        model_name="TranAD",
        explanation_name="SHAP",
    )
)

TranADSHAPFaithfulnessResults = pd.DataFrame(
    TranADSHAPFaithfulnessList
)

TranADSHAPFaithfulnessResults

,Model,Explanation,Index,Case Type,Label,Original Score,Perturbed Score,Score Difference,Faithfulness Score
0,TranAD,SHAP,2154,TN,0,0.005808,0.003987,-0.001821,0.313613
1,TranAD,SHAP,1647,TN,0,0.009400,0.008971,-0.000429,0.045656
2,TranAD,SHAP,161,TN,0,0.006775,0.006095,-0.000680,0.100317
3,TranAD,SHAP,1394,TN,0,0.007250,0.006782,-0.000468,0.064503
4,TranAD,SHAP,929,TN,0,0.004773,0.002836,-0.001937,0.405890
5,TranAD,SHAP,911,TN,0,0.006403,0.002838,-0.003565,0.556776
6,TranAD,SHAP,1528,TN,0,0.006568,0.005389,-0.001180,0.179604
7,TranAD,SHAP,193,TN,0,0.009590,0.009173,-0.000417,0.043499
8,TranAD,SHAP,383,TN,0,0.005922,0.004779,-0.001143,0.193059
9,TranAD,SHAP,1878,TN,0,0.006818,0.004012,-0.002806,0.411581


In [42]:
TranADSHAPFaithfulnessList = (
    FaithfulnessEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        temporal_importance_values=TranADSHAPImportanceValues,
        score_function=tranad_score_function,
        model_name="TranAD",
        explanation_name="SHAP",
    )
)

TranADSHAPFaithfulnessResults = pd.DataFrame(
    TranADSHAPFaithfulnessList
)

TranADSHAPFaithfulnessResults

,Model,Explanation,Index,Case Type,Label,Original Score,Perturbed Score,Score Difference,Faithfulness Score
0,TranAD,SHAP,2154,TN,0,0.005808,0.003987,-0.001821,0.313613
1,TranAD,SHAP,1647,TN,0,0.009400,0.008971,-0.000429,0.045656
2,TranAD,SHAP,161,TN,0,0.006775,0.006095,-0.000680,0.100317
3,TranAD,SHAP,1394,TN,0,0.007250,0.006782,-0.000468,0.064503
4,TranAD,SHAP,929,TN,0,0.004773,0.002836,-0.001937,0.405890
5,TranAD,SHAP,911,TN,0,0.006403,0.002838,-0.003565,0.556776
6,TranAD,SHAP,1528,TN,0,0.006568,0.005389,-0.001180,0.179604
7,TranAD,SHAP,193,TN,0,0.009590,0.009173,-0.000417,0.043499
8,TranAD,SHAP,383,TN,0,0.005922,0.004779,-0.001143,0.193059
9,TranAD,SHAP,1878,TN,0,0.006818,0.004012,-0.002806,0.411581


In [43]:
TranADSHAPFaithfulnessSummary = (
    TranADSHAPFaithfulnessResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Faithfulness Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADSHAPFaithfulnessSummary

,Model,Explanation,Case Type,mean,std,count
0,TranAD,SHAP,TN,0.231450,0.180905,10
1,TranAD,SHAP,TP,0.541593,0.224471,10


In [44]:
# Save the Results

TranADSHAPFaithfulnessResults.to_csv(
    FAITHFULNESS_DIR / "tranad_shap_faithfulness_results.csv",
    index=False,
)

TranADSHAPFaithfulnessSummary.to_csv(
    FAITHFULNESS_DIR / "tranad_shap_faithfulness_summary.csv",
    index=False,
)

# Final Faithfulness Evaluation Results

In [45]:
FaithfulnessSummary = pd.concat(
    [
        AnoFormerAttentionFaithfulnessSummary,
        AnoFormerSHAPFaithfulnessSummary,
        TranADAttentionFaithfulnessSummary,
        TranADSHAPFaithfulnessSummary,
    ],
    ignore_index=True,
)

FaithfulnessSummary

,Model,Explanation,Case Type,mean,std,count
0,AnoFormer,Attention,TN,0.294846,0.239207,10
1,AnoFormer,Attention,TP,0.456079,0.125415,10
2,AnoFormer,SHAP,TN,0.393123,0.221748,10
3,AnoFormer,SHAP,TP,0.376761,0.187494,10
4,TranAD,Attention,TN,0.173084,0.193090,10
5,TranAD,Attention,TP,0.509286,0.246041,10
6,TranAD,SHAP,TN,0.231450,0.180905,10
7,TranAD,SHAP,TP,0.541593,0.224471,10


In [46]:
FaithfulnessSummary.to_csv(
    FAITHFULNESS_DIR / "faithfulness_summary_all_methods.csv",
    index=False,
)